# 01 — Data Ingestion
## Bluestock MF Analytics — Day 1
**Tasks:** Load all 10 CSVs, fetch live NAV, validate AMFI codes


## Setup

In [ ]:
import pandas as pd
import numpy as np
import requests
import json
import time
from pathlib import Path

ROOT      = Path.cwd().parent
RAW       = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

print(f"Raw data directory: {RAW}")
print(f"Files found: {len(list(RAW.glob('*.csv')))}")


## Task 1 — Load all 10 CSV Datasets

In [ ]:
datasets = {}
csv_files = sorted(RAW.glob("*.csv"))

for path in csv_files:
    name = path.stem
    try:
        df = pd.read_csv(path, low_memory=False)
        datasets[name] = df
        print(f"✔ {name:<45} {df.shape}")
    except Exception as e:
        print(f"✘ {name}: {e}")

print(f"\nTotal datasets loaded: {len(datasets)}")


## Task 2 — Print Shape, DTypes, Head for each

In [ ]:
for name, df in list(datasets.items())[:10]:  # first 10 only
    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")
    print(f"Shape : {df.shape}")
    print(f"\nDtypes:\n{df.dtypes.to_string()}")
    print(f"\nHead:\n{df.head(3).to_string()}")


## Task 3 — Anomaly Detection

In [ ]:
anomalies = []
for name, df in list(datasets.items())[:10]:
    nulls = df.isnull().sum().sum()
    dups  = df.duplicated().sum()
    if nulls > 0 or dups > 0:
        anomalies.append({"dataset": name, "nulls": nulls, "duplicates": dups})
        print(f"⚠  {name}: {nulls} nulls, {dups} duplicates")
    else:
        print(f"✔  {name}: Clean")

print(f"\nTotal anomalies found in: {len(anomalies)} datasets")


## Task 4 — Fetch Live NAV from mfapi.in

In [ ]:
BASE_URL = "https://api.mfapi.in/mf"
HDFC_CODE = 125497

def fetch_nav(code, retries=3):
    for attempt in range(retries):
        try:
            resp = requests.get(f"{BASE_URL}/{code}", timeout=15)
            resp.raise_for_status()
            return resp.json()
        except Exception as e:
            print(f"  Attempt {attempt+1} failed: {e}")
            time.sleep(2)
    return None

data = fetch_nav(HDFC_CODE)
if data:
    df_hdfc = pd.DataFrame(data["data"])
    df_hdfc["scheme_code"] = HDFC_CODE
    df_hdfc["scheme_name"] = data["meta"]["scheme_name"]
    df_hdfc["date"] = pd.to_datetime(df_hdfc["date"], dayfirst=True, errors="coerce")
    df_hdfc["nav"]  = pd.to_numeric(df_hdfc["nav"], errors="coerce")
    df_hdfc = df_hdfc.sort_values("date").reset_index(drop=True)
    df_hdfc.to_csv(RAW / f"nav_hdfc_{HDFC_CODE}.csv", index=False)
    print(f"✔ HDFC Top 100 fetched: {len(df_hdfc):,} records")
    print(f"  Latest NAV: ₹{df_hdfc['nav'].iloc[-1]:.4f} as of {df_hdfc['date'].iloc[-1].date()}")


## Task 5 — Fetch 5 Key Schemes

In [ ]:
KEY_SCHEMES = {
    119551: "SBI Bluechip Fund Direct Growth",
    120503: "ICICI Prudential Bluechip Fund Direct Growth",
    118632: "Nippon India Large Cap Fund Direct Growth",
    119092: "Axis Bluechip Fund Direct Growth",
    120841: "Kotak Bluechip Fund Direct Growth",
}

all_nav = []
for code, name in KEY_SCHEMES.items():
    print(f"\nFetching: {name} ({code})")
    data = fetch_nav(code)
    if data:
        df = pd.DataFrame(data["data"])
        df["scheme_code"] = code
        df["scheme_name"] = data["meta"]["scheme_name"]
        df["date"] = pd.to_datetime(df["date"], dayfirst=True, errors="coerce")
        df["nav"]  = pd.to_numeric(df["nav"], errors="coerce")
        df = df.sort_values("date").reset_index(drop=True)
        df.to_csv(RAW / f"nav_{code}.csv", index=False)
        all_nav.append(df)
        print(f"  ✔ {len(df):,} records | Latest NAV: ₹{df['nav'].iloc[-1]:.4f}")
    time.sleep(0.5)

combined = pd.concat(all_nav, ignore_index=True)
combined.to_csv(RAW / "nav_five_schemes_combined.csv", index=False)
print(f"\n✔ Combined: {len(combined):,} records saved")


## Task 6 — Validate AMFI Codes

In [ ]:
import glob

fm_files  = list(RAW.glob("*fund_master*.csv")) + list(RAW.glob("*master*.csv"))
nav_files = list(RAW.glob("*nav_history*.csv"))

if fm_files and nav_files:
    fm  = pd.read_csv(fm_files[0])
    nav = pd.read_csv(nav_files[0])
    fm_col  = next((c for c in fm.columns  if "code" in c.lower()), None)
    nav_col = next((c for c in nav.columns if "code" in c.lower()), None)
    if fm_col and nav_col:
        fm_codes  = set(fm[fm_col].dropna().astype(int))
        nav_codes = set(nav[nav_col].dropna().astype(int))
        matched   = fm_codes & nav_codes
        missing   = fm_codes - nav_codes
        print(f"fund_master codes  : {len(fm_codes):,}")
        print(f"nav_history codes  : {len(nav_codes):,}")
        print(f"Matched            : {len(matched):,}")
        print(f"Missing in nav     : {len(missing):,}")
        print(f"Status             : {'✔ PASS' if not missing else '⚠ WARN'}")
else:
    print("⚠ fund_master or nav_history not found — place in data/raw/")

print("\n✔ Data Ingestion Complete!")
